# Digital Image Processing Lab — Lab Task 03
## Image Enhancement and Intensity Transformations

**Student:** Raja Wajiullah  
**Roll No.:** 2024-SE-28  
**Program:** Software Engineering (SE), 5th Semester  
**Instructor:** Engr. Fahad Nisar  
**Submission date:** 10 November 2026

> The uploaded draft notebook was reviewed and expanded: the assigned +40/−40 brightness values are used, and Otsu and adaptive Gaussian thresholding are included. Before running: place your personally selected JPG/PNG image (at least 300 × 300 pixels) in the same folder as this notebook; the default path is `image.png`. This notebook preserves the original image and uses safe arithmetic for intensity transformations.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

IMAGE_PATH = "image.png"  # Change to your own .jpg or .png file

img = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
if img is None:
    raise FileNotFoundError(
        f"Could not load {IMAGE_PATH!r}. Put your JPG/PNG beside this notebook "
        "or update IMAGE_PATH."
    )
if img.dtype != np.uint8:
    raise TypeError(f"Expected 8-bit grayscale uint8 image, got {img.dtype}")
if img.ndim != 2:
    raise ValueError(f"Expected a 2-D grayscale image, got shape {img.shape}")
if img.shape[0] < 300 or img.shape[1] < 300:
    raise ValueError(f"Image must be at least 300 x 300 pixels; got {img.shape}")

original = img.copy()  # Keep original unchanged
print("Shape:", original.shape)
print("dtype:", original.dtype)
print("Minimum:", int(original.min()))
print("Maximum:", int(original.max()))

plt.figure(figsize=(6, 5))
plt.imshow(original, cmap="gray", vmin=0, vmax=255)
plt.title("Original Grayscale Image")
plt.axis("off")
plt.show()


## Common helper: safe conversion

All arithmetic is performed in a wider type (`int16`, `int32`, or `float64`). Values are clipped to `[0, 255]` before conversion to `uint8`. This prevents unsigned 8-bit wraparound/overflow. Fractional values are truncated unless a task explicitly asks for rounding.

In [ ]:
def safe_uint8(values):
    return np.clip(values, 0, 255).astype(np.uint8)  # astype truncates fractions

def compare_images(title, result):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(original, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("Original")
    axes[1].imshow(result, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title(title)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## 1. Brightness adjustment

**Formula:** `s = r + β`, with β = +40 and β = −40. Positive β brightens and negative β darkens. Values outside the 8-bit range are clipped.

**Dry run:** Input `[30, 80, 128, 180, 240]`.

- β = +40: raw `[70, 120, 168, 220, 280]`; clipped/final `[70, 120, 168, 220, 255]`.
- β = −40: raw `[-10, 40, 88, 140, 200]`; clipped/final `[0, 40, 88, 140, 200]`.
- The 240 pixel becomes 255 in the brighter result and loses highlight distinction; 30 becomes 0 in the darker result and loses shadow distinction.

The loop implementation below must match NumPy exactly.

In [ ]:
beta_plus = safe_uint8(original.astype(np.int16) + 40)
beta_minus = safe_uint8(original.astype(np.int16) - 40)

# Required nested-loop implementation for +40
loop_plus = np.empty_like(original)
h, w = original.shape
for y in range(h):
    for x in range(w):
        value = int(original[y, x]) + 40
        loop_plus[y, x] = min(255, max(0, value))

print("NumPy and loop implementations match:", np.array_equal(beta_plus, loop_plus))
assert np.array_equal(beta_plus, loop_plus)
compare_images("Brightness +40", beta_plus)
compare_images("Brightness -40", beta_minus)

dry = np.array([30, 80, 128, 180, 240], dtype=np.int16)
print("Dry run +40 raw:", (dry + 40).tolist())
print("Dry run +40 final:", safe_uint8(dry + 40).tolist())
print("Dry run -40 raw:", (dry - 40).tolist())
print("Dry run -40 final:", safe_uint8(dry - 40).tolist())


## 2. Linear contrast from zero

**Formula:** `s = αr + β`, where α = 1.5 and β = 0. Multiplying intensities by 1.5 increases separation from zero, but values above 255 saturate.

**Dry run:** `1.5 × [30, 80, 128, 180, 240] = [45, 120, 192, 270, 360]`. After clipping and truncation: `[45, 120, 192, 255, 255]`. The last two pixels saturate at 255. This operation increases both intensity and contrast relative to black; it is not centered around mid-gray.

In [ ]:
linear_raw = original.astype(np.float32) * 1.5
linear_contrast = safe_uint8(linear_raw)
compare_images("Linear contrast α=1.5, β=0", linear_contrast)

dry = np.array([30, 80, 128, 180, 240], dtype=np.float64)
print("Raw 1.5r:", (1.5 * dry).tolist())
print("Clipped/truncated:", safe_uint8(1.5 * dry).tolist())


## 3. Contrast centered at 128

**Formula:** `s = (r − 128)α + 128`.

For α = 1.5, values below 128 move farther down and values above 128 move farther up, increasing contrast around mid-gray. For α = 0.5, values move toward 128, reducing contrast. The input 128 remains exactly 128 for either α.

**Dry run** for `[30, 80, 128, 180, 240]`:
- α = 1.5 raw `[-19, 56, 128, 206, 296]`; final `[0, 56, 128, 206, 255]`.
- α = 0.5 raw `[79, 104, 128, 154, 184]`; final `[79, 104, 128, 154, 184]`.


In [ ]:
r = original.astype(np.float32)
centered_15 = safe_uint8((r - 128) * 1.5 + 128)
centered_05 = safe_uint8((r - 128) * 0.5 + 128)
compare_images("Centered contrast α=1.5", centered_15)
compare_images("Centered contrast α=0.5", centered_05)

dry = np.array([30, 80, 128, 180, 240], dtype=np.float64)
for alpha in (1.5, 0.5):
    raw = (dry - 128) * alpha + 128
    print(f"α={alpha}: raw={raw.tolist()}, final={safe_uint8(raw).tolist()}")


## 4. Image negative

**Formula:** `s = 255 − r`. A negative reverses the grayscale scale: dark pixels become bright and bright pixels become dark. It can help inspect bright structures against a dark background (or faint dark structures against a bright background), for example in some radiographic or microscopy images.

**Dry run:** `[30, 80, 128, 180, 240]` becomes `[225, 175, 127, 75, 15]`. Applying the negative again returns `[30, 80, 128, 180, 240]` exactly.

In [ ]:
negative = 255 - original.astype(np.int16)
negative = negative.astype(np.uint8)
compare_images("Image negative", negative)
negative_twice = 255 - negative.astype(np.int16)
print("Applying negative twice restores original:", np.array_equal(original, negative_twice.astype(np.uint8)))
dry = np.array([30, 80, 128, 180, 240], dtype=np.int16)
print("Negative:", (255 - dry).tolist())
print("Negative twice:", (255 - (255 - dry)).tolist())


## 5. Gamma correction

**Formula:** `s = 255(r/255)^γ`. Normalize first, apply the power, then scale by 255. With this convention, γ = 0.5 brightens mid/dark values; γ = 2.0 darkens them. Both 0 and 255 remain fixed. Keep full precision through the calculation and truncate only the final output.

**Dry run:** input `[0, 64, 128, 192, 255]`.
- γ = 0.5: normalized `[0, .250980, .501961, .752941, 1]`; power `[0, .500979, .708492, .867722, 1]`; scaled `[0, 127.749755, 180.665437, 221.269067, 255]`; final `[0, 127, 180, 221, 255]`.
- γ = 2.0: power `[0, .062991, .251965, .566920, 1]`; scaled `[0, 16.062745, 64.250980, 144.564706, 255]`; final `[0, 16, 64, 144, 255]`.

A rounded slide example can differ by one or more intensity levels because this task specifies truncation of the full-precision result rather than intermediate rounding.

In [ ]:
def gamma_transform(image, gamma):
    normalized = image.astype(np.float64) / 255.0
    powered = normalized ** gamma
    scaled = 255.0 * powered
    return np.trunc(scaled).clip(0, 255).astype(np.uint8)

gamma_05 = gamma_transform(original, 0.5)
gamma_20 = gamma_transform(original, 2.0)
compare_images("Gamma γ=0.5 (brightens)", gamma_05)
compare_images("Gamma γ=2.0 (darkens)", gamma_20)

dry = np.array([0, 64, 128, 192, 255], dtype=np.float64)
for gamma in (0.5, 2.0):
    norm = dry / 255.0
    power = norm ** gamma
    scaled = 255.0 * power
    final = np.trunc(scaled).astype(np.uint8)
    print(f"γ={gamma}")
    print(" normalized:", norm.tolist())
    print(" power:", power.tolist())
    print(" scaled:", scaled.tolist())
    print(" final:", final.tolist())
assert gamma_05.min() >= 0 and gamma_05.max() <= 255
assert gamma_20.min() >= 0 and gamma_20.max() <= 255


## 6. Grayscale histogram

A grayscale histogram counts the number of pixels at each intensity. Plot 256 bins over `[0, 256]`.

**Dry run:** `A = [[0, 50, 50, 100], [100, 150, 200, 255]]`, total pixels = 8.
Nonzero frequencies: 0:1, 50:2, 100:2, 150:1, 200:1, 255:1. Probabilities are respectively 0.125, 0.25, 0.25, 0.125, 0.125, 0.125. A histogram discards spatial coordinates, so different images with the same intensity counts can have identical histograms.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(original.ravel(), bins=256, range=(0, 256), color="gray")
plt.xlim(0, 256)
plt.xlabel("Intensity")
plt.ylabel("Pixel count")
plt.title("Original Grayscale Histogram (256 bins)")
plt.grid(alpha=0.2)
plt.show()

A = np.array([[0, 50, 50, 100], [100, 150, 200, 255]], dtype=np.uint8)
values, counts = np.unique(A, return_counts=True)
print("Total pixels:", A.size)
print("Intensity : frequency : probability")
for value, count in zip(values, counts):
    print(int(value), ":", int(count), ":", count / A.size)


## 7. Global histogram equalization

Use `cv2.equalizeHist` to redistribute global intensity values using the cumulative distribution. It can improve global contrast when intensities occupy a narrow range, but it may amplify noise and cannot adapt independently to local lighting.

**Dry run:** `[50, 50, 100, 150]`; frequencies 2, 1, 1; cumulative counts 2, 3, 4; CDF 0.5, 0.75, 1.0; `CDFmin = 0.5`.
- Direct mapping `round(255 × CDF)` gives `[128, 191, 255]`, hence output list `[128, 128, 191, 255]`.
- Normalized mapping `round(255 × (CDF − CDFmin)/(1 − CDFmin))` gives `[0, 128, 255]`, hence `[0, 0, 128, 255]`.
- OpenCV uses the normalized-CDF style mapping (with integer implementation details), so compare against the second list. The required dry-run rounding is nearest integer with .5 upward.

In [ ]:
equalized = cv2.equalizeHist(original)
compare_images("Global histogram equalization", equalized)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(original.ravel(), bins=256, range=(0, 256), color="gray")
axes[0].set_title("Original histogram")
axes[1].hist(equalized.ravel(), bins=256, range=(0, 256), color="gray")
axes[1].set_title("Equalized histogram")
for ax in axes:
    ax.set_xlim(0, 256)
    ax.set_xlabel("Intensity")
    ax.set_ylabel("Pixel count")
plt.tight_layout()
plt.show()

dry = [50, 50, 100, 150]
unique, freq = np.unique(dry, return_counts=True)
cum = np.cumsum(freq)
cdf = cum / len(dry)
cdf_min = cdf[np.flatnonzero(freq)[0]]
direct = np.floor(255 * cdf + 0.5).astype(int)
normalized = np.floor(255 * (cdf - cdf_min) / (1 - cdf_min) + 0.5).astype(int)
map_direct = dict(zip(unique.tolist(), direct.tolist()))
map_normalized = dict(zip(unique.tolist(), normalized.tolist()))
print("unique:", unique.tolist(), "frequency:", freq.tolist())
print("cumulative:", cum.tolist(), "CDF:", cdf.tolist(), "CDFmin:", cdf_min)
print("Direct mapped list:", [map_direct[v] for v in dry])
print("Normalized mapped list:", [map_normalized[v] for v in dry])


## 8. CLAHE

CLAHE (Contrast Limited Adaptive Histogram Equalization) equalizes local tiles and limits contrast amplification. It is useful when illumination varies across the image. `tileGridSize=(8, 8)` means 8 tiles horizontally and 8 vertically, not tiles of 8 × 8 pixels. Neighboring tile mappings are interpolated to avoid abrupt boundaries.

**Dry run:** four-bin histogram `[6, 2, 0, 0]`, absolute cap 4. Clip to `[4, 2, 0, 0]`; excess = 2. Redistribute equally across four bins (allow fractions): `[4.5, 2.5, 0.5, 0.5]`. Total remains 8. New CDF = `[0.5625, 0.875, 0.9375, 1.0]`. This simplified fractional redistribution is not OpenCV's integer redistribution, and the absolute cap is not the `clipLimit` parameter.

In [ ]:
clahe_operator = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clahe_image = clahe_operator.apply(original)
compare_images("CLAHE clipLimit=2.0, tileGridSize=(8,8)", clahe_image)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(equalized, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Global equalization")
axes[1].imshow(clahe_image, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("CLAHE")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

clipped = np.array([4, 2, 0, 0], dtype=float)
excess = 2.0
redistributed = clipped + excess / 4
new_cdf = np.cumsum(redistributed) / redistributed.sum()
print("Clipped counts:", clipped.tolist())
print("Excess:", excess)
print("Redistributed counts:", redistributed.tolist())
print("New CDF:", new_cdf.tolist())


## 9. Fixed binary thresholding

Use `cv2.THRESH_BINARY` at T = 127. A pixel is white (255) only when `r > T`; otherwise it is black (0). Report unique values.

**Dry run:** `[0, 80, 127, 128, 180, 255]`.
- T = 127 → `[0, 0, 0, 255, 255, 255]`, 3 white pixels.
- T = 180 → `[0, 0, 0, 0, 0, 255]`, 1 white pixel.
The strict greater-than condition means a pixel equal to T is black.

In [ ]:
_, fixed_mask = cv2.threshold(original, 127, 255, cv2.THRESH_BINARY)
plt.figure(figsize=(6, 5))
plt.imshow(fixed_mask, cmap="gray", vmin=0, vmax=255)
plt.title("Fixed Binary Threshold T=127")
plt.axis("off")
plt.show()
print("Unique values:", np.unique(fixed_mask).tolist())

dry = np.array([0, 80, 127, 128, 180, 255], dtype=np.uint8)
for T in (127, 180):
    result = np.where(dry > T, 255, 0)
    print(f"T={T}: {result.tolist()}, white count={(result == 255).sum()}")


## 10. Otsu thresholding

Otsu selects a global threshold that maximizes between-class variance `w0 w1 (μ0 − μ1)²`. Here classes are split at `r ≤ T` and `r > T`; if maxima tie, choose the smallest T.

**Dry run:** `[0, 1, 4, 5]`.
- T=0: `w0=.25, μ0=0; w1=.75, μ1=3.3333; score=2.0833`.
- T=1: `w0=.5, μ0=.5; w1=.5, μ1=4.5; score=4`.
- T=2: same score 4.
- T=3: same score 4.
- T=4: `w0=.75, μ0=1.6667; w1=.25, μ1=5; score=2.0833`.
The tied maximum occurs at 1, 2, and 3; choose the smallest, T=1. Predicted mask `[0, 0, 255, 255]`.

In [ ]:
otsu_value, otsu_mask = cv2.threshold(
    original, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)
print("Otsu threshold returned by OpenCV:", otsu_value)
print("Otsu mask unique values:", np.unique(otsu_mask).tolist())
plt.figure(figsize=(6, 5))
plt.imshow(otsu_mask, cmap="gray", vmin=0, vmax=255)
plt.title(f"Otsu Binary Mask (T={otsu_value:.0f})")
plt.axis("off")
plt.show()

dry = np.array([0, 1, 4, 5], dtype=np.uint8)
scores = []
for T in range(5):
    c0, c1 = dry[dry <= T], dry[dry > T]
    if len(c0) == 0 or len(c1) == 0:
        score = 0.0
        w0 = len(c0) / len(dry); w1 = len(c1) / len(dry)
        mu0 = float(c0.mean()) if len(c0) else float("nan")
        mu1 = float(c1.mean()) if len(c1) else float("nan")
    else:
        w0, w1 = len(c0)/len(dry), len(c1)/len(dry)
        mu0, mu1 = float(c0.mean()), float(c1.mean())
        score = w0*w1*(mu0-mu1)**2
    scores.append((T, w0, w1, mu0, mu1, score))
for row in scores:
    print("T,w0,w1,mu0,mu1,score =", row)
best = max(row[-1] for row in scores)
best_T = min(row[0] for row in scores if np.isclose(row[-1], best))
print("Selected smallest maximizing T:", best_T)
print("Predicted mask:", np.where(dry > best_T, 255, 0).tolist())


## 11. Adaptive Gaussian thresholding

Use `cv2.adaptiveThreshold` with Gaussian weighting, binary output, `blockSize=11`, and `C=2`. Each pixel gets a local threshold based on its neighborhood; local thresholds can work better than one global threshold when lighting is uneven.

**Dry run:** neighborhood `[[100,100,100],[100,120,140],[140,140,140]]`; Gaussian weights `[[1,2,1],[2,4,2],[1,2,1]]/16`. Weighted sum = 1920/16 = 120, so T = mean − 2 = 118. Center pixel 120 is white because 120 > 118. A pixel exactly equal to T is black because the rule is strict `r > T`.

In [ ]:
adaptive_mask = cv2.adaptiveThreshold(
    original, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY, 11, 2
)
plt.figure(figsize=(6, 5))
plt.imshow(adaptive_mask, cmap="gray", vmin=0, vmax=255)
plt.title("Adaptive Gaussian Threshold (blockSize=11, C=2)")
plt.axis("off")
plt.show()
print("Unique values:", np.unique(adaptive_mask).tolist())

neighborhood = np.array([[100,100,100],[100,120,140],[140,140,140]], dtype=float)
weights = np.array([[1,2,1],[2,4,2],[1,2,1]], dtype=float) / 16
weighted_mean = float(np.sum(neighborhood * weights))
T_local = weighted_mean - 2
center_pixel = neighborhood[1,1]
print("Weighted mean:", weighted_mean)
print("Local threshold:", T_local)
print("Center pixel classification:", 255 if center_pixel > T_local else 0)
print("Pixel equal to threshold classification:", 255 if T_local > T_local else 0)


## Final observations and conclusion

- **Brightness** adds or subtracts a constant and shifts the intensity level; **contrast** changes the separation between intensity values.
- **Clipping** limits values to the valid 8-bit range `[0,255]`. Without a wider arithmetic type, unsigned 8-bit arithmetic can wrap around and produce incorrect values.
- **Gamma convention:** with `s=255(r/255)^γ`, γ below 1 brightens midtones and γ above 1 darkens midtones.
- **Global enhancement** (global histogram equalization) applies one mapping to the whole image; it can improve overall contrast but may amplify noise and may not fix spatially uneven illumination.
- **Local enhancement** (CLAHE) operates on tiles and interpolates neighboring mappings, often helping under uneven lighting while limiting excessive contrast amplification.
- **Thresholding:** fixed thresholding is simple when a suitable global cutoff is known; Otsu automatically selects a global threshold for a histogram with separable classes; adaptive thresholding is useful when illumination changes across the image.

### Completion checklist
1. Set `IMAGE_PATH` to a personally selected image of at least 300 × 300 pixels.
2. Run every cell from a restarted kernel.
3. Confirm the NumPy and nested-loop brightness results match.
4. Inspect all displayed images and histograms; write brief observations based on your actual image.
5. Save this notebook with the required filename and submit it with the Word report and input image.
